# Rupesh Man Pradhan — GIS V01.4: **real satellite/street map** with zero external JavaScript dependencies

**Run All** to verify the original XLSX, extract all 266 researcher coordinates, render the real satellite/street/topographic map with a self-contained HTML renderer, and produce the audit files.

**What's fixed:** V01.3 depended on remote Leaflet/Bootstrap/CDN scripts which may not load inside Colab's sandbox iframe. This notebook embeds the entire HTML/CSS/JavaScript map viewer without a CDN, so **markers, toolbar, zoom and pan render even if the map tiles are unavailable**. Satellite/street/terrain tiles themselves are still downloaded live from Esri or OSM HOT to your browser; they cannot be guaranteed when network access is blocked. An on-screen failure warning provides links to inspect coordinates in Google Maps. Nothing is silently passed off as a geographic basemap.

The 265 records near Pokhara and 1 anomalous Ward 33 record are preserved (150 engineered, 116 non-engineered). One cross-category duplicate is flagged. Coordinates are assumed to be EPSG:4326 for visual review only, not certified. Source file ID and SHA-256 unchanged; no thesis content changed. Review building-location privacy and project-sharing permissions before publishing.


In [ ]:
# 1 — Configuration and exact-source acquisition
import os, io, re, csv, html, json, hashlib, zipfile, collections
from pathlib import Path
from xml.etree import ElementTree as ET
from IPython.display import display, HTML

SOURCE_DRIVE_FILE_ID = "1piJxk-V3xUl_CB0NxEnc5g1G2hbp1WK0"
SOURCE_DRIVE_FILE_NAME = "GOOGLE CO ORDINATE__ORIGINAL__20261012.xlsx"
EXPECTED_SHA256 = "19982a5a718a81fe71d65c30935714ea6dae88db0a7670a2927b61de60ec88c7"
OUTPUT_DIR = Path("/content/RMP_GIS_V01" if Path("/content").exists() else "./RMP_GIS_V01")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# For offline re-runs, supply an exact local copy here; in Colab, the canonical
# unchanged .xlsx is downloaded using the signed-in researcher's Google account.
local_candidates = [
    Path("/mnt/data/GOOGLE CO ORDINATE.xlsx"),
    Path("/content/GOOGLE CO ORDINATE.xlsx"),
    Path.cwd() / "GOOGLE CO ORDINATE.xlsx"
]
source_path = next((p for p in local_candidates if p.exists()), None)

if source_path is None:
    from google.colab import auth
    auth.authenticate_user()
    import google.auth
    from googleapiclient.discovery import build
    from googleapiclient.http import MediaIoBaseDownload
    creds, _ = google.auth.default(scopes=["https://www.googleapis.com/auth/drive.readonly"])
    service = build("drive", "v3", credentials=creds, cache_discovery=False)
    req = service.files().get_media(fileId=SOURCE_DRIVE_FILE_ID)
    source_path = OUTPUT_DIR / SOURCE_DRIVE_FILE_NAME
    with source_path.open("wb") as fp:
        downloader = MediaIoBaseDownload(fp, req, chunksize=1024*1024)
        done = False
        while not done:
            _, done = downloader.next_chunk()

source_sha256 = hashlib.sha256(source_path.read_bytes()).hexdigest()
assert source_sha256 == EXPECTED_SHA256, (
    "Input hash differs from the governed original. STOP — check source/version "
    f"(expected {EXPECTED_SHA256}, got {source_sha256})"
)
print("Input:", source_path)
print("Verified SHA256:", source_sha256)

In [ ]:
# 2 — Lossless source-row extraction from XLSX using Python's standard XML/ZIP APIs
# No guesswork, interpolation, coordinate correction, or spreadsheet data rewriting.
NS = {"x":"http://schemas.openxmlformats.org/spreadsheetml/2006/main"}
WARD_RE = re.compile(r"\bward\b\s*(?:no\.?|number|#)?\s*0*(\d{1,2})\b", re.I)
PAIR_RE = re.compile(r"^\s*([+-]?\d+(?:\.\d+)?)\s*,\s*([+-]?\d+(?:\.\d+)?)\s*$")
def read_xlsx_cells(xlsx_path, worksheet_no):
    with zipfile.ZipFile(xlsx_path) as archive:
        strings=[]
        if "xl/sharedStrings.xml" in archive.namelist():
            tree=ET.fromstring(archive.read("xl/sharedStrings.xml"))
            for si in tree.findall("x:si",NS):
                strings.append("".join(x.text or "" for x in si.iterfind(".//x:t",NS)))
        tree=ET.fromstring(archive.read(f"xl/worksheets/sheet{worksheet_no}.xml"))
        for row in tree.findall(".//x:sheetData/x:row",NS):
            rownum=int(row.attrib["r"])
            cells={}
            for cell in row.findall("x:c",NS):
                col=re.match(r"[A-Z]+",cell.attrib["r"]).group(0)
                typ=cell.attrib.get("t","")
                value=cell.find("x:v",NS)
                if typ=="inlineStr":
                    inline=cell.find("x:is",NS)
                    s="".join(n.text or "" for n in inline.iterfind(".//x:t",NS)) if inline is not None else ""
                elif value is None:
                    continue
                elif typ=="s":
                    s=strings[int(value.text)]
                else:
                    s=value.text or ""
                cells[col]=s
            yield rownum,cells

records=[]
headings=[]
for sheet_no, sheet_name, category, src_col in [
    (1,"Non_Engineered_Building_Coordin","Non-engineered","B"),
    (2,"Engineered_Building_Coodrinates","Engineered","A")
]:
    ward=None
    for source_row,cells in read_xlsx_cells(source_path,sheet_no):
        text=(cells.get(src_col) or "").strip()
        if not text:
            continue
        wm=WARD_RE.search(text)
        if wm:
            ward=int(wm.group(1))
            headings.append({"sheet":sheet_name,"ward":ward,"source_row":source_row})
            continue
        m=PAIR_RE.match(text)
        if not m:
            raise ValueError(f"Unexpected source text {sheet_name}!{src_col}{source_row}: {text!r}")
        if ward is None:
            raise ValueError(f"Missing preceding Ward label: {sheet_name}!{src_col}{source_row}")
        lat,lon=map(float,m.groups())
        is_pokhara_cluster = (27.8 <= lat <= 28.7 and 83.5 <= lon <= 84.6)
        point_id=f"RMP-{'ENG' if category=='Engineered' else 'NON'}-W{ward:02d}-R{source_row:03d}"
        records.append({
            "point_id":point_id,"category":category,"ward":ward,
            "latitude":lat,"longitude":lon,"sheet":sheet_name,
            "source_row":source_row,"raw_coordinate":text,
            "map_review_status": "PLOT_PRELIMINARY_UNVERIFIED_CRS" if is_pokhara_cluster else "HOLD_OUTSIDE_STUDY_AREA",
            "duplicate_flag":"","original_file_id":SOURCE_DRIVE_FILE_ID
        })
coord_lookup=collections.defaultdict(list)
for r in records:
    coord_lookup[(r["latitude"],r["longitude"])].append(r)
for r in records:
    if len(coord_lookup[(r["latitude"],r["longitude"])])>1:
        r["duplicate_flag"]="EXACT_CROSS_CATEGORY_DUPLICATE"

counts=collections.Counter(r["category"] for r in records)
holds=[r for r in records if r["map_review_status"]=="HOLD_OUTSIDE_STUDY_AREA"]
plotted=[r for r in records if r["map_review_status"]!="HOLD_OUTSIDE_STUDY_AREA"]
assert len(records)==266 and counts["Engineered"]==150 and counts["Non-engineered"]==116
assert len(holds)==1 and holds[0]["point_id"]=="RMP-ENG-W33-R207"
assert len(headings)==59
assert sum(bool(r["duplicate_flag"]) for r in records)==2
print("Records:",len(records), "| Engineered:",counts["Engineered"], "| Non-engineered:",counts["Non-engineered"])
print("Main Pokhara points:",len(plotted), "| Separate QA-hold layer:",len(holds))
print("Exact cross-category duplicate rows:",sum(bool(r["duplicate_flag"]) for r in records))

In [ ]:
# 3 — Preserve lossless QA, GeoJSON and provenance before plotting
import csv, json
version="GIS-V01.4"
geojson_path=OUTPUT_DIR/"RMP_all_266_points_QA_v0.1.4.geojson"
qa_path=OUTPUT_DIR/"RMP_coordinates_all_266_QA_v0.1.4.csv"
manifest_path=OUTPUT_DIR/"RMP_GIS_V01_4_run_manifest.json"
features=[]
for p in records:
    features.append({"type":"Feature", "id":p["point_id"],
        "geometry":{"type":"Point","coordinates":[p["longitude"],p["latitude"]]},
        "properties":{k:p[k] for k in ["category","ward","sheet","source_row",
            "raw_coordinate","map_review_status","duplicate_flag","original_file_id"]}})
geojson={"type":"FeatureCollection","features":features,
    "metadata":{"source_drive_id":SOURCE_DRIVE_FILE_ID,
                "source_sha256":source_sha256,
                "crs_note":"WGS84/EPSG:4326 assumed from source numeric lat/lon; not certified",
                "publication_status":"PRIVATE_REVIEW_ONLY"}}
geojson_path.write_text(json.dumps(geojson,indent=2),encoding="utf-8")
fields=["point_id","category","ward","latitude","longitude","raw_coordinate",
        "sheet","source_row","map_review_status","duplicate_flag","original_file_id"]
with qa_path.open("w",newline="",encoding="utf-8") as fp:
    wr=csv.DictWriter(fp,fieldnames=fields,extrasaction="ignore")
    wr.writeheader(); wr.writerows(records)
print("Source records",len(records), "| accepted for main view",len(plotted),
    "| held outside region",len(holds))
print("Coordinate QA CSV:",qa_path)
print("GeoJSON (all 266 records):",geojson_path)


In [ ]:
# 4 — Real Esri satellite, streets & topo tiles. No external Leaflet, JS libraries or CDN.
# Basemap images still require permitted browser network access; red health message on failure.
import html, math, json
from IPython.display import HTML, display
from string import Template
WEB_TEMPLATE='<!doctype html>\n<html lang="en"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1">\n<title>Rupesh GIS V01.4 — 266 survey points, actual satellite/street tiles</title>\n<style>\n:root{color-scheme:light;--bg:#0c2035;--blue:#2779ee;--orange:#ff873a;--ink:#122239}\n*{box-sizing:border-box}html,body{padding:0;margin:0;background:#eef3f8;color:var(--ink);font-family:system-ui,-apple-system,Segoe UI,Arial,sans-serif}\n.header{background:var(--bg);color:white;padding:13px 20px}.header h1{font-size:19px;line-height:1.35;margin:0 0 3px;font-weight:750}.header p{margin:0;font-size:12px;color:#d1e2f5}\n.toolbar{background:#fff;display:flex;align-items:center;flex-wrap:wrap;gap:9px;padding:10px 14px;border-bottom:1px solid #cbd5e1;box-shadow:0 1px 4px #0002}\n.toolbar label{display:flex;align-items:center;gap:5px;font-size:12px;font-weight:600}\nselect,button.action{background:white;border:1px solid #9aadc0;border-radius:6px;color:#182c3f;padding:7px 9px;font:inherit;font-size:12px}\nbutton.action{cursor:pointer}button.action:hover{background:#ecf5ff}select:focus,button:focus,input:focus{outline:2px solid #377ede;outline-offset:1px}\n.legend{display:flex;gap:11px;font-size:12px;align-items:center}.dot{display:inline-block;width:10px;height:10px;border-radius:50%;margin-right:4px}\n.frame{position:relative;width:100%;height:650px;overflow:hidden;user-select:none;touch-action:none;cursor:grab;background:#e9edf1}\n.frame:active{cursor:grabbing}\n#tile-layer,#marker-layer{position:absolute;inset:0;pointer-events:none}\n.tile{position:absolute;width:256px;height:256px;object-fit:cover;border:0;max-width:none;pointer-events:none;image-rendering:auto}\n.point{position:absolute;transform:translate(-50%,-50%);width:11px;height:11px;border:1.5px solid white;border-radius:50%;padding:0;box-shadow:0 0 3px #0009;cursor:pointer;pointer-events:auto}\n.point.engineered{background:#2677ee}.point.non-engineered{background:#ff823c}.point.flagged{border:2px solid #e11d48;box-shadow:0 0 0 3px #fecdd3}\n#popup{position:absolute;z-index:12;top:64px;left:15px;max-width:min(350px,calc(100% - 30px));background:white;padding:14px;border-radius:8px;box-shadow:0 3px 16px #0005;font-size:13px;display:none}\n#popup a{color:#0757af}.popup-head{display:flex;justify-content:space-between;align-items:center;gap:10px;margin-bottom:8px}\n#popup-close{cursor:pointer;border:0;background:#dbeafe;border-radius:4px;padding:3px 7px}\n.coord-grid{position:absolute;inset:0;pointer-events:none;z-index:0;background-image:linear-gradient(to right,#8fa4b833 1px,transparent 1px),linear-gradient(to bottom,#8fa4b833 1px,transparent 1px);background-size:128px 128px}\n#tile-layer{z-index:1}#marker-layer{z-index:3}\n.map-controls{position:absolute;z-index:8;top:12px;right:12px;display:grid;gap:5px}\n.map-controls button{width:36px;height:36px;background:#fff;border:1px solid #afbed0;border-radius:6px;font-size:20px;cursor:pointer;box-shadow:0 1px 4px #0002}\n.status{position:relative;padding:11px 16px;font-size:13px;line-height:1.5;background:#f5f9ff;border-top:1px solid #cfdbeb;display:flex;gap:15px;justify-content:space-between;flex-wrap:wrap}\n#tile-health{font-weight:650}.ok{color:#12633c}.warn{color:#985000}.bad{color:#a62525}.subtle{color:#516174}\n#fail-notice{display:none;position:absolute;z-index:7;left:15px;bottom:15px;max-width:380px;background:#fffef8;border:1px solid #edbe65;box-shadow:0 1px 12px #0002;border-radius:7px;padding:12px;font-size:12px}\n#fail-notice a{color:#0757af}.credits{font-size:11px;color:#44556b;background:white;padding:6px 14px;border-top:1px solid #d9e1e8}\n@media(max-width:820px){.header{padding:9px 12px}.header h1{font-size:16px}.toolbar{padding:8px;gap:6px}.frame{height:550px}.toolbar label{font-size:11px}}\n</style></head><body>\n<header class="header"><h1>Rupesh Man Pradhan · Pokhara building survey</h1>\n<p>GIS V01.4 · 266 original observations · 265 mapped in Pokhara + 1 outlier held for inspection · provisional WGS84</p></header>\n<section class="toolbar">\n<label>Actual map <select id="basemap"><option value="satellite">Satellite — Esri</option><option value="streets">Streets — Esri</option><option value="terrain">Topographic — Esri</option><option value="hot">Streets — OSM Humanitarian</option></select></label>\n<label>Ward <select id="ward"><option value="all">All wards</option>__OPTIONS__</select></label>\n<label><input type="checkbox" id="eng" checked><span class="dot" style="background:#2677ee"></span>Engineered (150)</label>\n<label><input type="checkbox" id="non" checked><span class="dot" style="background:#ff823c"></span>Non-engineered (116)</label>\n<button class="action" id="fit">Pokhara extent</button><button class="action" id="outlier">Inspect Ward 33 outlier</button>\n<span class="subtle" id="count" aria-live="polite">265 main points</span>\n</section>\n<main id="map" class="frame" aria-label="Interactive Pokhara map with survey locations">\n<div class="coord-grid"></div><div id="tile-layer">__TILES__</div><div id="marker-layer">__MARKERS__</div>\n<div class="map-controls"><button title="Zoom in" id="zoom-in">+</button><button title="Zoom out" id="zoom-out">−</button></div>\n<div id="popup"></div><div id="fail-notice" role="alert"><strong>Basemap tiles cannot load.</strong><p>The coordinates are preserved and visible, but your Colab/network is blocking the map imagery service. Try the other map providers above, or open the satellite view directly: <a id="fallback-link" href="https://www.google.com/maps/@28.2,83.98,11z/data=!3m1!1e3" target="_blank" rel="noopener">Open Google Maps satellite</a>.</p></div>\n</main>\n<div class="status"><span id="tile-health" role="status" aria-live="polite">Checking live map imagery…</span><span class="subtle">Drag to pan · Mouse wheel / ± to zoom · Click a point to inspect its source row</span></div>\n<div class="credits" id="credits">Satellite imagery © Esri, Maxar, Earthstar Geographics and contributors. Point-location classification comes only from Rupesh\'s workbook; use privately until consent and CRS are verified.</div>\n<script>\n(function(){\n\'use strict\';\nconst records=__RECORDS__;\nconst providers=__PROVIDERS__;\nconst map=document.getElementById(\'map\');\nconst tiles=document.getElementById(\'tile-layer\');\nconst markers=document.getElementById(\'marker-layer\');\nconst popup=document.getElementById(\'popup\');\nconst tileStatus=document.getElementById(\'tile-health\');\nconst fail=document.getElementById(\'fail-notice\');\nconst select=document.getElementById(\'basemap\');\nconst ward=document.getElementById(\'ward\');\nconst eng=document.getElementById(\'eng\');const non=document.getElementById(\'non\');\nconst count=document.getElementById(\'count\');\nconst main=records.filter(p=>!p.map_review_status.includes(\'HOLD\'));\nconst holds=records.filter(p=>p.map_review_status.includes(\'HOLD\'));\nconst b={south:Math.min(...main.map(p=>p.latitude)),north:Math.max(...main.map(p=>p.latitude)),west:Math.min(...main.map(p=>p.longitude)),east:Math.max(...main.map(p=>p.longitude))};\nlet center={lat:(b.south+b.north)/2,lon:(b.west+b.east)/2};let zoom=12;\nlet holdActive=false;let loadToken=0;let drag=null;let loaded=0,failed=0,planned=0;\nlet provider=providers[0];\nfunction xy(lat,lon,z){const scale=256*Math.pow(2,z);const a=Math.max(-85.0511,Math.min(85.0511,lat))*Math.PI/180;return {x:(lon+180)/360*scale,y:(1-Math.asinh(Math.tan(a))/Math.PI)/2*scale};}\nfunction lnglat(px,py,z){const scale=256*Math.pow(2,z),n=Math.PI-2*Math.PI*py/scale;return {lat:Math.atan(Math.sinh(n))*180/Math.PI,lon:px/scale*360-180};}\nfunction url(p,x,y,z){return p.url.replace(\'{x}\',x).replace(\'{y}\',y).replace(\'{z}\',z)}\nfunction health(message,state){tileStatus.textContent=message;tileStatus.className=state||\'\'}\nfunction fitBounds(){const w=map.clientWidth,h=map.clientHeight;for(let z=16;z>=7;z--){const a=xy(b.north,b.west,z),d=xy(b.south,b.east,z);if(d.x-a.x<w*.85&&d.y-a.y<h*.85){zoom=z;break;}}center={lat:(b.south+b.north)/2,lon:(b.west+b.east)/2};holdActive=false;draw()}\nfunction pop(p){const raw=String(p.raw_coordinate).replace(/</g,\'&lt;\').replace(/>/g,\'&gt;\');const name=String(p.point_id).replace(/</g,\'&lt;\');const mapurl=`https://www.google.com/maps?q=${p.latitude},${p.longitude}&t=k`;\npopup.innerHTML=`<div class="popup-head"><strong>${name}</strong><button id="popup-close" aria-label="Close popup">×</button></div><b>${p.category}</b> · Ward ${p.ward}<br>${p.latitude.toFixed(8)}°, ${p.longitude.toFixed(8)}°<br>Original sheet: ${p.sheet}<br>Source row: ${p.source_row}<br>Original: ${raw}<br><a href="${mapurl}" target="_blank" rel="noopener">See this coordinate in Google Maps</a>${p.duplicate_flag?\'<p style="color:#b91c1c">Duplicate coordinate across categories: review</p>\':\'\'}${p.map_review_status.includes(\'HOLD\')?\'<p style="color:#b91c1c">QA hold: outside primary Pokhara extent.</p>\':\'\'}`;\npopup.style.display=\'block\';document.getElementById(\'popup-close\').onclick=()=>popup.style.display=\'none\';}\nfunction draw(){const token=++loadToken;loaded=0;failed=0;planned=0;fail.style.display=\'none\';\nconst w=map.clientWidth,h=map.clientHeight;const cp=xy(center.lat,center.lon,zoom);const left=cp.x-w/2,top=cp.y-h/2;\nconst frag=document.createDocumentFragment();const zcount=Math.pow(2,zoom);\nfor(let y=Math.floor(top/256);y<=Math.floor((top+h)/256);y++)for(let x=Math.floor(left/256);x<=Math.floor((left+w)/256);x++){\nif(y<0||y>=zcount)continue;const normalized=((x%zcount)+zcount)%zcount;\nconst image=document.createElement(\'img\');image.className=\'tile\';image.alt=\'\';image.draggable=false;\nimage.style.left=(x*256-left)+\'px\';image.style.top=(y*256-top)+\'px\';planned++;\nimage.onload=()=>{if(token!==loadToken)return;loaded++;if(loaded===1){health(\'Actual \'+provider.label+\' map tiles loaded (\'+planned+\' tiles requested).\',\'ok\');fail.style.display=\'none\';}};\nimage.onerror=()=>{if(token!==loadToken)return;failed++;if(failed>=Math.min(3,planned)&&loaded===0){health(provider.label+\' tiles blocked or unavailable. Choose another provider.\',\'bad\');fail.style.display=\'block\';}};\nimage.src=url(provider,normalized,y,zoom);frag.appendChild(image);\n}\ntiles.replaceChildren(frag);if(planned===0){health(\'Outside tile coverage\',\'warn\');}\nconst pts=holdActive?records:main;const markerFrag=document.createDocumentFragment();let shown=0;\nfor(const p of pts){if(ward.value!==\'all\'&&String(p.ward)!==ward.value)continue;if(p.category===\'Engineered\'&&!eng.checked||p.category===\'Non-engineered\'&&!non.checked)continue;\nconst v=xy(p.latitude,p.longitude,zoom),x=v.x-left,y=v.y-top;\nif(x<-20||y<-20||x>w+20||y>h+20)continue;\nconst button=document.createElement(\'button\');button.className=\'point \'+(p.category===\'Engineered\'?\'engineered\':\'non-engineered\')+(p.duplicate_flag?\' flagged\':\'\');\nbutton.style.left=x+\'px\';button.style.top=y+\'px\';button.title=p.point_id+\' · Ward \'+p.ward+\' · \'+p.category;button.setAttribute(\'aria-label\',button.title);\nbutton.onclick=(ev)=>{ev.stopPropagation();pop(p)};markerFrag.appendChild(button);shown++;\n}\nmarkers.replaceChildren(markerFrag);count.textContent=shown+\' visible records · \'+(holdActive?\'all 266 available\':\'1 QA hold preserved\');\nconst copyright=provider.credit;document.getElementById(\'credits\').textContent=copyright+\'. Source-coordinate CRS remains unverified; research-use only.\';\nconst timerToken=token;setTimeout(()=>{if(timerToken===loadToken&&loaded===0){health(\'No \'+provider.label+\' tiles loaded in this browser yet; points remain visible.\',\'bad\');fail.style.display=\'block\';}},8500);\n}\nselect.onchange=()=>{provider=providers.find(p=>p.id===select.value)||providers[0];draw()};\nward.onchange=()=>{holdActive=false;fitBounds();};eng.onchange=draw;non.onchange=draw;\ndocument.getElementById(\'fit\').onclick=()=>{ward.value=\'all\';eng.checked=true;non.checked=true;fitBounds();};\ndocument.getElementById(\'outlier\').onclick=()=>{holdActive=true;ward.value=\'all\';zoom=11;center={lat:holds[0].latitude,lon:holds[0].longitude};draw();pop(holds[0]);};\ndocument.getElementById(\'zoom-in\').onclick=()=>{zoom=Math.min(18,zoom+1);draw()};\ndocument.getElementById(\'zoom-out\').onclick=()=>{zoom=Math.max(3,zoom-1);draw()};\nmap.addEventListener(\'pointerdown\',e=>{if(e.target.closest(\'button,select,input,a\'))return;drag={x:e.clientX,y:e.clientY,center:xy(center.lat,center.lon,zoom)};map.setPointerCapture(e.pointerId)});\nmap.addEventListener(\'pointerup\',e=>{if(!drag)return;const cc=lnglat(drag.center.x+(drag.x-e.clientX),drag.center.y+(drag.y-e.clientY),zoom);center={lat:cc.lat,lon:cc.lon};drag=null;draw();});\nmap.addEventListener(\'pointercancel\',()=>drag=null);\nmap.addEventListener(\'wheel\',e=>{e.preventDefault();zoom=Math.min(18,Math.max(3,zoom+(e.deltaY<0?1:-1)));draw()},{passive:false});\nnew ResizeObserver(()=>draw()).observe(map);fitBounds();\n})();\n</script></body></html>\n'
PROVIDERS=[{'id': 'satellite', 'label': 'Satellite imagery', 'type': 'ESRI', 'url': 'https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}', 'credit': 'Satellite imagery © Esri, Maxar, Earthstar Geographics and contributors'}, {'id': 'streets', 'label': 'Street map', 'type': 'ESRI', 'url': 'https://server.arcgisonline.com/ArcGIS/rest/services/World_Street_Map/MapServer/tile/{z}/{y}/{x}', 'credit': 'Street map © Esri and contributors'}, {'id': 'terrain', 'label': 'Terrain / topography', 'type': 'ESRI', 'url': 'https://server.arcgisonline.com/ArcGIS/rest/services/World_Topo_Map/MapServer/tile/{z}/{y}/{x}', 'credit': 'Topography © Esri and contributors'}, {'id': 'hot', 'label': 'OSM Humanitarian streets', 'type': 'HOT', 'url': 'https://a.tile.openstreetmap.fr/hot/{z}/{x}/{y}.png', 'credit': '© OpenStreetMap contributors; Humanitarian style © OSM France'}]

def make_tile_map(plotted, holds):
    records=plotted+holds
    minlat,maxlat=min(p['latitude'] for p in plotted),max(p['latitude'] for p in plotted)
    minlon,maxlon=min(p['longitude'] for p in plotted),max(p['longitude'] for p in plotted)
    zoom=12; tile=256; width,height=1120,680
    def xy(lat,lon,z):
        s=tile*2**z;a=math.radians(max(-85.05112878,min(85.05112878,lat)))
        return ((lon+180)/360*s,(1-math.asinh(math.tan(a))/math.pi)/2*s)
    lat=(minlat+maxlat)/2;lon=(minlon+maxlon)/2
    cx,cy=xy(lat,lon,zoom);left,top=cx-width/2,cy-height/2
    initial=[]
    for ty in range(int(top//tile),int((top+height)//tile)+2):
        for tx in range(int(left//tile),int((left+width)//tile)+2):
            if tx<0 or ty<0 or tx>=2**zoom or ty>=2**zoom:continue
            url=PROVIDERS[0]['url'].replace('{z}',str(zoom)).replace('{x}',str(tx)).replace('{y}',str(ty))
            initial.append(f'<img class="tile" loading="eager" decoding="async" draggable="false" style="left:{tx*tile-left:.3f}px;top:{ty*tile-top:.3f}px" src="{html.escape(url,quote=True)}" alt="Satellite map tile">')
    pins=[]
    for p in plotted:
        x,y=xy(p['latitude'],p['longitude'],zoom)
        name=html.escape(p['point_id'],quote=True)
        category='engineered' if p['category']=='Engineered' else 'non-engineered'
        pins.append(f'<button class="point {category}" style="left:{x-left:.2f}px;top:{y-top:.2f}px" title="{name}"></button>')
    opts=''.join(f'<option value="{w}">Ward {w:02d}</option>' for w in sorted(set(p['ward'] for p in records)))
    rows_json=json.dumps([{k:p[k] for k in ['point_id','category','ward','latitude','longitude','raw_coordinate','sheet','source_row','map_review_status','duplicate_flag']} for p in records],separators=(',',':')).replace('</','<\/')
    return (WEB_TEMPLATE.replace('__OPTIONS__',opts).replace('__TILES__',''.join(initial)).replace('__MARKERS__',''.join(pins))
             .replace('__RECORDS__',rows_json).replace('__PROVIDERS__',json.dumps(PROVIDERS,separators=(',',':'))))

web_html=make_tile_map(plotted,holds)
map_path=OUTPUT_DIR/'RMP_building_coordinates_ALL_266_RESILIENT_BASEMAP_v0.1.4.html'
map_path.write_text(web_html,encoding='utf-8')
assert web_html.count('<button class="point ')==265
assert 'World_Imagery/MapServer' in web_html and 'World_Street_Map/MapServer' in web_html
assert '<script src=' not in web_html and 'leaflet' not in web_html.lower()
print('SELF-CONTAINED MAP:', map_path)
print('REAL BASEMAPS: satellite, street, terrain, HOT — live tile images only; inline JS/CSS, no external libraries.')
print('266 preserved = 265 in Pokhara + 1 anomalous QA hold.')
# IFrame executes inline map JS in a Colab output frame; srcdoc carries the entire standalone file.
iframe=f'<iframe title="Rupesh GIS: real satellite and street map" width="100%" height="830" sandbox="allow-scripts allow-same-origin allow-popups" style="border:1px solid #cbd5e1;border-radius:8px" srcdoc="{html.escape(web_html,quote=True)}"></iframe>'
display(HTML(iframe))
print('If imagery tiles show a warning, choose another provider or open the HTML outside Colab; marker/QA data remain visible.')


In [ ]:
# 5 — Emergency coordinate-only fallback image (NOT a basemap) (requires no web access or basemap key)
import matplotlib.pyplot as plt
fig2,ax=plt.subplots(figsize=(11.8,7.8))
for cat,col in [('Engineered','#2563eb'),('Non-engineered','#f97316')]:
    ps=[p for p in plotted if p['category']==cat]
    ax.scatter([p['longitude'] for p in ps],[p['latitude'] for p in ps],
        s=34,alpha=.83,c=col,label=f'{cat} — {len(ps)} main points',edgecolors='white',linewidths=.35)
ax.set_xlabel('Longitude (decimal degrees; WGS84 not independently certified)')
ax.set_ylabel('Latitude (decimal degrees; WGS84 not independently certified)')
ax.set_title('Rupesh — main Pokhara survey points (265)\n1 anomalous point separately preserved in QA')
ax.grid(alpha=.23)
ax.set_aspect(1/max(.001,__import__('math').cos(__import__('math').radians(28.2))))
ax.legend(loc='best')
fig2.tight_layout()
static_path=OUTPUT_DIR/'RMP_building_coordinates_265_OFFLINE_preview_v0.1.4.png'
fig2.savefig(static_path,dpi=185,bbox_inches='tight')
plt.show()
print('Static PNG:',static_path)


In [ ]:
# 6 — Run manifest (only after all outputs are materialized)
import platform,sys,hashlib,json
manifest={
 'researcher':'Rupesh Man Pradhan','version':'GIS-V01.4',
 'repair':'LIBRARY_FREE_INLINE_JS_WITH_HTML_TILE_IMAGES_AND_FAILURE_WARNING',
 'source_drive_id':SOURCE_DRIVE_FILE_ID,'source_filename':SOURCE_DRIVE_FILE_NAME,
 'source_sha256':source_sha256,'record_count':len(records),
 'engineered_count':counts['Engineered'],'non_engineered_count':counts['Non-engineered'],
 'main_extent_points':len(plotted),'outlier_hold_count':len(holds),
 'outlier_point_ids':[p['point_id'] for p in holds],
 'overlap_records':sum(bool(p['duplicate_flag']) for p in records),
 'crs_status':'WGS84_ASSUMED_NOT_CERTIFIED',
 'research_state':'PRIVATE_REVIEW_ONLY','tile_availability':'PENDING_USER_BROWSER_VERIFICATION','basemap_providers':['Esri World Imagery','Esri World Street Map','Esri World Topographic Map','OSM Humanitarian tile server'],'python_version':sys.version.split()[0],
 'artifacts':[]
}
for f in [map_path,static_path,qa_path,geojson_path]:
    manifest['artifacts'].append({'name':f.name,'size_bytes':f.stat().st_size,
                  'sha256':hashlib.sha256(f.read_bytes()).hexdigest()})
manifest_path.write_text(json.dumps(manifest,indent=2),encoding='utf-8')
print('Verified artifact inventory:')
for f in manifest['artifacts']:print(f['name'],f['size_bytes'],'bytes',f['sha256'][:12])
print('Manifest:',manifest_path)


In [ ]:
# 7 — OPTIONAL controlled Drive upload. Off by default; avoid duplicates.
UPLOAD_RESULTS_TO_DRIVE=False
TARGET_FOLDERS={
 map_path.name:'1iAP-wSc4nSqjNW8odXaFg-ymQNNb46zj',
 static_path.name:'1iAP-wSc4nSqjNW8odXaFg-ymQNNb46zj',
 geojson_path.name:'1I3NkTIVG8pEZX-x87Kv-Mn8o5-wNdnaU',
 qa_path.name:'1sPcfq-LADyGnOZcYDK-5QyMpFKl1pWj6',
 manifest_path.name:'1sPcfq-LADyGnOZcYDK-5QyMpFKl1pWj6',
}
if UPLOAD_RESULTS_TO_DRIVE:
    from google.colab import auth
    auth.authenticate_user()
    import google.auth
    from googleapiclient.discovery import build
    from googleapiclient.http import MediaFileUpload
    creds,_=google.auth.default(scopes=['https://www.googleapis.com/auth/drive.file'])
    service=build('drive','v3',credentials=creds,cache_discovery=False)
    for name,dest in TARGET_FOLDERS.items():
        query=f"name = '{name}' and '{dest}' in parents and trashed = false"
        matches=service.files().list(q=query,fields='files(id,name)',page_size=100).execute().get('files',[])
        if matches:
            print('Existing output SKIPPED:',name,[m['id'] for m in matches]);continue
        result=service.files().create(body={'name':name,'parents':[dest]},
           media_body=MediaFileUpload(str(OUTPUT_DIR/name),resumable=True),fields='id,name').execute()
        print('Uploaded',result)
else:
    print('Automatic Drive re-upload is OFF. Use the verified Drive outputs for this release.')


### Why the previous notebook was blank
V01.3 depended on Leaflet and other externally loaded JS/CSS; the Colab iframe could show static HTML controls without the map library or its map initialization. V01.4 contains the entire viewer's JavaScript and CSS inline; the satellite/street tiles are plain `<img>` requests from selected providers. Even if tiles are denied, all 266 records remain intact and a clear visible warning is shown. No third-party credentials are embedded. Precise surveyed building locations are private until permission is confirmed.